### dbt_exploration_2

In [10]:
import os
import pandas as pd
import snowflake.connector
from dotenv import load_dotenv
from IPython.display import display

pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_rows', 100)

load_dotenv()

conn = snowflake.connector.connect(
    account=os.getenv("SNOWFLAKE_ACCOUNT"),
    user=os.getenv("SNOWFLAKE_USER"),
    password=os.getenv("SNOWFLAKE_PASSWORD"),
    role=os.getenv("SNOWFLAKE_ROLE"),
    warehouse=os.getenv("SNOWFLAKE_WAREHOUSE"),
    database=os.getenv("SNOWFLAKE_DATABASE"),
)

def run_query(sql: str) -> pd.DataFrame:
    cur = conn.cursor()
    cur.execute(sql)
    rows = cur.fetchall()
    cols = [desc[0] for desc in cur.description]
    cur.close()
    return pd.DataFrame(rows, columns=cols)

print("Connected!")

Connected!


## 1. Raw Table Profiling

For each source table, we:
1. Discover every key present in `RAW_PAYLOAD` dynamically using `OBJECT_KEYS()`
2. Count how often each key is non-null across all rows
3. Pull one representative sample value per key

This tells us the true shape of each source's payload — including fields the docs may not mention.

In [14]:
def profile_raw_source(database: str, schema: str, table: str) -> pd.DataFrame:
    full_table = f"{database}.{schema}.{table}"

    # Pull all payloads into Python as JSON strings
    raw_df = run_query(f"SELECT RAW_PAYLOAD::STRING AS payload FROM {full_table}")
    total_rows = len(raw_df)

    # Parse JSON and discover all keys in Python
    import json
    all_keys = set()
    parsed = []
    for row in raw_df['PAYLOAD']:
        try:
            obj = json.loads(row)
            parsed.append(obj)
            all_keys.update(obj.keys())
        except Exception:
            parsed.append({})

    # For each key: count non-nulls and grab a sample value
    records = []
    for key in sorted(all_keys):
        values = [p.get(key) for p in parsed]
        non_null = [v for v in values if v is not None]
        sample = str(non_null[0])[:120] if non_null else None
        records.append({
            'PAYLOAD_KEY': key,
            'NON_NULL_COUNT': len(non_null),
            'total_rows': total_rows,
            'population_pct': round(len(non_null) / total_rows * 100, 1),
            'SAMPLE_VALUE': sample,
        })

    return (
        pd.DataFrame(records)
        .sort_values('population_pct', ascending=False)
        .reset_index(drop=True)
    )

In [15]:
print('Profiling JSearch raw payload...')
jsearch_profile = profile_raw_source('RAW', 'JSEARCH', 'SRC_POSTINGS')

print(f"Total rows: {jsearch_profile['total_rows'].iloc[0]}")
print(f"Unique payload keys discovered: {len(jsearch_profile)}")
print()
display(jsearch_profile.drop(columns=['total_rows']))

Profiling JSearch raw payload...
Total rows: 136
Unique payload keys discovered: 34



,PAYLOAD_KEY,NON_NULL_COUNT,population_pct,SAMPLE_VALUE
0,apply_options,136,100.0,[{'apply_link': 'https://www.linkedin.com/jobs/view/data-analyst-new-york-at...
1,job_description,136,100.0,Jobright is a next-generation AI job search platform built to make career na...
2,job_publisher,136,100.0,LinkedIn
3,job_posted_at_timestamp,136,100.0,1779260400
4,job_posted_at_datetime_utc,136,100.0,2026-05-20T07:00:00.000Z
5,job_posted_at,136,100.0,18 hours ago
6,job_location,136,100.0,"New York, NY"
7,job_id,136,100.0,n7ha1uBvkMfQoGz5AAAAAA==
8,job_highlights,136,100.0,{}
9,job_google_link,136,100.0,https://www.google.com/search?q=jobs&gl=us&hl=en&udm=8#vhid=vt%3D20/docid%3D...


In [16]:
print('Profiling TheirStack raw payload...')
theirstack_profile = profile_raw_source('RAW', 'THEIRSTACK', 'SRC_POSTINGS')

print(f"Total rows: {theirstack_profile['total_rows'].iloc[0]}")
print(f"Unique payload keys discovered: {len(theirstack_profile)}")
print()
display(theirstack_profile.drop(columns=['total_rows']))

Profiling TheirStack raw payload...
Total rows: 18
Unique payload keys discovered: 48



,PAYLOAD_KEY,NON_NULL_COUNT,population_pct,SAMPLE_VALUE
0,latitude,18,100.0,40.71427
1,id,18,100.0,698550590
2,keyword_slugs,18,100.0,"['job-descriptions', 'sensors-test-measurement', 'reporting-and-disclosure',..."
3,cities,18,100.0,[]
4,location,18,100.0,"New York, NY"
5,locations,18,100.0,"[{'address': None, 'admin1_code': 'NY', 'admin1_name': 'New York', 'admin2_c..."
6,long_location,18,100.0,"New York, NY"
7,longitude,18,100.0,-74.00597
8,manager_roles,18,100.0,[]
9,matching_phrases,18,100.0,[]


In [17]:
print('Profiling Built In raw payload...')
builtin_profile = profile_raw_source('RAW', 'BUILTIN', 'SRC_POSTINGS')

print(f"Total rows: {builtin_profile['total_rows'].iloc[0]}")
print(f"Unique payload keys discovered: {len(builtin_profile)}")
print()
display(builtin_profile.drop(columns=['total_rows']))

Profiling Built In raw payload...
Total rows: 18
Unique payload keys discovered: 19



,PAYLOAD_KEY,NON_NULL_COUNT,population_pct,SAMPLE_VALUE
0,@context,18,100.0,https://schema.org
1,employmentType,18,100.0,FULL_TIME
2,title,18,100.0,Investment Quant & Data Analyst
3,source_url,18,100.0,https://www.builtinnyc.com/job/investment-quant-data-analyst/9550551
4,scraped_at,18,100.0,2026-05-31T18:12:20.445986+00:00
5,industry,18,100.0,"['Fintech', 'Payments', 'Financial Services']"
6,identifier,18,100.0,"{'@type': 'PropertyValue', 'name': 'Resolution Life', 'value': '9550551'}"
7,@type,18,100.0,JobPosting
8,hiringOrganization,18,100.0,"{'@type': 'Organization', 'logo': {'@type': 'ImageObject', 'representativeOf..."
9,directApply,18,100.0,False


In [18]:
comparison = (
    jsearch_profile[['PAYLOAD_KEY', 'population_pct']].rename(columns={'population_pct': 'jsearch_%'})
    .merge(
        theirstack_profile[['PAYLOAD_KEY', 'population_pct']].rename(columns={'population_pct': 'theirstack_%'}),
        on='PAYLOAD_KEY', how='outer'
    )
    .merge(
        builtin_profile[['PAYLOAD_KEY', 'population_pct']].rename(columns={'population_pct': 'builtin_%'}),
        on='PAYLOAD_KEY', how='outer'
    )
    .fillna(0.0)
    .sort_values('jsearch_%', ascending=False)
    .reset_index(drop=True)
)

print("Field coverage comparison across raw sources (population %)")
display(comparison)

Field coverage comparison across raw sources (population %)


,PAYLOAD_KEY,jsearch_%,theirstack_%,builtin_%
0,job_employment_types,100.0,0.0,0.0
1,job_apply_link,100.0,0.0,0.0
2,job_posted_at_datetime_utc,100.0,0.0,0.0
3,job_posted_at_timestamp,100.0,0.0,0.0
4,job_title,100.0,100.0,0.0
5,job_description,100.0,0.0,0.0
6,job_posted_at,100.0,0.0,0.0
7,job_google_link,100.0,0.0,0.0
8,job_highlights,100.0,0.0,0.0
9,job_id,100.0,0.0,0.0


## 2. Enrichment Table Profiling

Column-level population rates for `ENRICHED.PUBLIC.JOB_ENRICHMENT` across all records.

This tells us which enrichment fields the LLM is reliably filling vs. leaving null.

In [19]:
# Pull the full enrichment table once — reused for all profiling below
enrichment_df = run_query("SELECT * FROM ENRICHED.PUBLIC.JOB_ENRICHMENT")
print(f"Total enriched records: {len(enrichment_df)}")
print(f"Columns: {enrichment_df.columns.tolist()}")

Total enriched records: 172
Columns: ['JOB_ID', 'SOURCE', 'INFERRED_SENIORITY', 'IS_TITLE_INFLATED', 'INFLATION_REASONING', 'ROLE_ARCHETYPE', 'WORK_FOCUS', 'TECH_STACK_REQUIRED', 'TECH_STACK_PREFERRED', 'PARADIGMS_REQUIRED', 'PARADIGMS_PREFERRED', 'DEGREE_REQUIREMENT', 'YEARS_REQUIRED_MIN', 'YEARS_REQUIRED_MAX', 'SALARY_MIN', 'SALARY_MAX', 'CONFIDENCE_SCORE', 'ENRICHED_AT', 'MODEL_VERSION']


In [20]:
def profile_enrichment_table(df: pd.DataFrame) -> pd.DataFrame:
    total_rows = len(df)
    
    records = []
    for col in df.columns:
        non_null = df[col].notna().sum()
        sample = df[col].dropna().iloc[0] if non_null > 0 else None
        records.append({
            'COLUMN_NAME': col,
            'NON_NULL_COUNT': int(non_null),
            'total_rows': total_rows,
            'population_pct': round(non_null / total_rows * 100, 1),
            'SAMPLE_VALUE': str(sample)[:120] if sample is not None else None,
        })
    
    return pd.DataFrame(records).reset_index(drop=True)


def profile_enrichment_by_source(source_name: str, full_df: pd.DataFrame) -> pd.DataFrame:
    df = full_df[full_df['SOURCE'].str.upper() == source_name.upper()].copy()
    total_rows = len(df)

    skip_cols = {'JOB_ID', 'SOURCE', 'ENRICHED_AT'}
    
    records = []
    for col in df.columns:
        if col in skip_cols:
            continue
        non_null = df[col].notna().sum()
        sample = df[col].dropna().iloc[0] if non_null > 0 else None
        records.append({
            'COLUMN_NAME': col,
            'NON_NULL_COUNT': int(non_null),
            'total_rows': total_rows,
            'population_pct': round(non_null / total_rows * 100, 1),
            'SAMPLE_VALUE': str(sample)[:120] if sample is not None else None,
        })
    
    return pd.DataFrame(records).reset_index(drop=True)

In [21]:
print("Enrichment table — all sources:")
display(profile_enrichment_table(enrichment_df).drop(columns=['total_rows']))

Enrichment table — all sources:


,COLUMN_NAME,NON_NULL_COUNT,population_pct,SAMPLE_VALUE
0,JOB_ID,172,100.0,9526440
1,SOURCE,172,100.0,builtin
2,INFERRED_SENIORITY,172,100.0,mid
3,IS_TITLE_INFLATED,172,100.0,False
4,INFLATION_REASONING,52,30.2,The title 'Senior Data Analyst' suggests a higher level of experience than t...
5,ROLE_ARCHETYPE,172,100.0,data_analyst
6,WORK_FOCUS,172,100.0,build analytical foundations for media decisions
7,TECH_STACK_REQUIRED,172,100.0,"[\n ""sql"",\n ""snowflake"",\n ""tableau"",\n ""sigma"",\n ""lookers"",\n ""powe..."
8,TECH_STACK_PREFERRED,172,100.0,"[\n ""python"",\n ""r""\n]"
9,PARADIGMS_REQUIRED,172,100.0,"[\n ""data visualization"",\n ""data storytelling"",\n ""data management"",\n ..."


## 3. Enrichment Quality by Source

The same column-level profiling, but split by source.

Key things to look for:
- **Built In** — expect degraded extraction quality due to raw HTML in descriptions
- **TheirStack** — smaller sample size, watch for systematic nulls
- **JSearch** — largest sample, should be the baseline

In [22]:
source_counts = enrichment_df.groupby('SOURCE').size().reset_index(name='record_count')
print("Enriched record counts by source:")
display(source_counts)

Enriched record counts by source:


,SOURCE,record_count
0,builtin,18
1,jsearch:Analytics Engineer in New York,60
2,jsearch:Data Analyst in New York,76
3,theirstack,18


In [24]:
for source in ['jsearch:Analytics Engineer in New York', 'jsearch:Data Analyst in New York', 'theirstack', 'builtin']:
    print(f"\n{'='*50}")
    print(f"{source.upper()} enrichment quality:")
    result = profile_enrichment_by_source(source, enrichment_df)
    print(f"Records: {result['total_rows'].iloc[0]}")
    display(result.drop(columns=['total_rows']))


JSEARCH:ANALYTICS ENGINEER IN NEW YORK enrichment quality:
Records: 60


,COLUMN_NAME,NON_NULL_COUNT,population_pct,SAMPLE_VALUE
0,INFERRED_SENIORITY,60,100.0,senior
1,IS_TITLE_INFLATED,60,100.0,False
2,INFLATION_REASONING,27,45.0,The title 'Senior Software Engineer' implies a higher seniority than the 4 y...
3,ROLE_ARCHETYPE,60,100.0,software_engineer
4,WORK_FOCUS,60,100.0,design and build backend services and apis
5,TECH_STACK_REQUIRED,60,100.0,"[\n ""python"",\n ""sql"",\n ""airflow"",\n ""dbt""\n]"
6,TECH_STACK_PREFERRED,60,100.0,"[\n ""fastapi"",\n ""flask"",\n ""django"",\n ""aws""\n]"
7,PARADIGMS_REQUIRED,60,100.0,"[\n ""data governance"",\n ""data quality""\n]"
8,PARADIGMS_PREFERRED,60,100.0,"[\n ""ci/cd"",\n ""automated testing"",\n ""operational reliability""\n]"
9,DEGREE_REQUIREMENT,60,100.0,none



JSEARCH:DATA ANALYST IN NEW YORK enrichment quality:
Records: 76


,COLUMN_NAME,NON_NULL_COUNT,population_pct,SAMPLE_VALUE
0,INFERRED_SENIORITY,76,100.0,mid
1,IS_TITLE_INFLATED,76,100.0,False
2,INFLATION_REASONING,24,31.6,The title 'Senior Pricing Analytics Analyst' suggests a higher level of expe...
3,ROLE_ARCHETYPE,76,100.0,data_analyst
4,WORK_FOCUS,76,100.0,analyze performance and drive operational improvements
5,TECH_STACK_REQUIRED,76,100.0,"[\n ""sql""\n]"
6,TECH_STACK_PREFERRED,76,100.0,"[\n ""python""\n]"
7,PARADIGMS_REQUIRED,76,100.0,"[\n ""quantitative analysis"",\n ""data visualization""\n]"
8,PARADIGMS_PREFERRED,76,100.0,"[\n ""dashboarding""\n]"
9,DEGREE_REQUIREMENT,76,100.0,equivalent_ok



THEIRSTACK enrichment quality:
Records: 18


,COLUMN_NAME,NON_NULL_COUNT,population_pct,SAMPLE_VALUE
0,INFERRED_SENIORITY,18,100.0,mid
1,IS_TITLE_INFLATED,18,100.0,False
2,INFLATION_REASONING,0,0.0,NaN
3,ROLE_ARCHETYPE,18,100.0,data_analyst
4,WORK_FOCUS,18,100.0,"design, develop and maintain oracle database applications"
5,TECH_STACK_REQUIRED,18,100.0,"[\n ""oracle"",\n ""pl/sql"",\n ""sql"",\n ""sql developer"",\n ""toad""\n]"
6,TECH_STACK_PREFERRED,18,100.0,[]
7,PARADIGMS_REQUIRED,18,100.0,"[\n ""agile development"",\n ""database design"",\n ""data modeling"",\n ""perf..."
8,PARADIGMS_PREFERRED,18,100.0,[]
9,DEGREE_REQUIREMENT,18,100.0,none



BUILTIN enrichment quality:
Records: 18


,COLUMN_NAME,NON_NULL_COUNT,population_pct,SAMPLE_VALUE
0,INFERRED_SENIORITY,18,100.0,mid
1,IS_TITLE_INFLATED,18,100.0,False
2,INFLATION_REASONING,1,5.6,The title 'Senior Data Analyst' suggests a higher level of experience than t...
3,ROLE_ARCHETYPE,18,100.0,data_analyst
4,WORK_FOCUS,18,100.0,build analytical foundations for media decisions
5,TECH_STACK_REQUIRED,18,100.0,"[\n ""sql"",\n ""snowflake"",\n ""tableau"",\n ""sigma"",\n ""lookers"",\n ""powe..."
6,TECH_STACK_PREFERRED,18,100.0,"[\n ""python"",\n ""r""\n]"
7,PARADIGMS_REQUIRED,18,100.0,"[\n ""data visualization"",\n ""data storytelling"",\n ""data management"",\n ..."
8,PARADIGMS_PREFERRED,18,100.0,"[\n ""analytics processes"",\n ""automation opportunities"",\n ""reporting bes..."
9,DEGREE_REQUIREMENT,18,100.0,none


In [25]:
def get_source_pct(source_name: str) -> pd.DataFrame:
    return (
        profile_enrichment_by_source(source_name, enrichment_df)
        [['COLUMN_NAME', 'population_pct']]
        .rename(columns={'population_pct': f'{source_name}_%'})
    )

enrichment_comparison = (
    get_source_pct('jsearch')
    .merge(get_source_pct('theirstack'), on='COLUMN_NAME')
    .merge(get_source_pct('builtin'),    on='COLUMN_NAME')
)

enrichment_comparison['builtin_gap'] = (
    enrichment_comparison[['jsearch_%', 'theirstack_%']].mean(axis=1)
    - enrichment_comparison['builtin_%']
).round(1)

print("Enrichment quality by source — population % per column")
print("(builtin_gap = how far Built In lags behind the JSearch/TheirStack average)")
display(
    enrichment_comparison.sort_values('builtin_gap', ascending=False).reset_index(drop=True)
)

Enrichment quality by source — population % per column
(builtin_gap = how far Built In lags behind the JSearch/TheirStack average)


/var/folders/lc/2rldm7cs08zcbz67h_7qc6vw0000gn/T/ipykernel_72084/978289292.py:35: RuntimeWarning: invalid value encountered in scalar divide
  'population_pct': round(non_null / total_rows * 100, 1),


,COLUMN_NAME,jsearch_%,theirstack_%,builtin_%,builtin_gap
0,YEARS_REQUIRED_MAX,NaN,50.0,38.9,11.1
1,INFERRED_SENIORITY,NaN,100.0,100.0,0.0
2,IS_TITLE_INFLATED,NaN,100.0,100.0,0.0
3,ROLE_ARCHETYPE,NaN,100.0,100.0,0.0
4,WORK_FOCUS,NaN,100.0,100.0,0.0
5,TECH_STACK_REQUIRED,NaN,100.0,100.0,0.0
6,TECH_STACK_PREFERRED,NaN,100.0,100.0,0.0
7,PARADIGMS_REQUIRED,NaN,100.0,100.0,0.0
8,PARADIGMS_PREFERRED,NaN,100.0,100.0,0.0
9,DEGREE_REQUIREMENT,NaN,100.0,100.0,0.0


## 4. Bonus: Value Distribution Checks

Quick frequency tables for the key categorical enrichment fields — useful for validating
that the LLM is using the expected enum values and not hallucinating novel ones.

In [26]:
categorical_fields = [
    ('INFERRED_SENIORITY', 'Seniority distribution'),
    ('ROLE_ARCHETYPE',     'Role archetype distribution'),
    ('DEGREE_REQUIREMENT', 'Degree requirement distribution'),
    ('IS_TITLE_INFLATED',  'Title inflation flags'),
]

for col, label in categorical_fields:
    if col not in enrichment_df.columns:
        print(f"{label}: column not found, skipping")
        continue
    df = (
        enrichment_df[col]
        .value_counts(dropna=False)
        .rename_axis('value')
        .reset_index(name='count')
    )
    df['pct'] = (df['count'] / df['count'].sum() * 100).round(1)
    print(f"\n{label}:")
    display(df)


Seniority distribution:


,value,count,pct
0,mid,95,55.2
1,senior,48,27.9
2,entry,29,16.9



Role archetype distribution:


,value,count,pct
0,data_analyst,93,54.1
1,data_engineer,29,16.9
2,analytics_engineer,22,12.8
3,hybrid,22,12.8
4,software_engineer,6,3.5



Degree requirement distribution:


,value,count,pct
0,bachelors,80,46.5
1,none,56,32.6
2,equivalent_ok,28,16.3
3,masters,8,4.7



Title inflation flags:


,value,count,pct
0,False,120,69.8
1,True,52,30.2


In [27]:
if 'CONFIDENCE_SCORE' in enrichment_df.columns:
    conf_df = (
        enrichment_df['CONFIDENCE_SCORE']
        .dropna()
        .round(1)
        .value_counts()
        .sort_index()
        .rename_axis('score_bucket')
        .reset_index(name='count')
    )
    print("Confidence score distribution (LLM self-assessment):")
    display(conf_df)

Confidence score distribution (LLM self-assessment):


,score_bucket,count
0,0.5,1
1,0.7,4
2,0.8,47
3,0.9,120


In [28]:
salary_df = (
    enrichment_df
    .groupby('SOURCE')
    .apply(lambda g: pd.Series({
        'total_enriched':   len(g),
        'llm_salary_found': g['SALARY_MIN'].notna().sum(),
        'llm_salary_pct':   round(g['SALARY_MIN'].notna().sum() / len(g) * 100, 1),
    }))
    .reset_index()
)

print("LLM salary extraction rate by source:")
display(salary_df)

LLM salary extraction rate by source:


,SOURCE,total_enriched,llm_salary_found,llm_salary_pct
0,builtin,18.0,14.0,77.8
1,jsearch:Analytics Engineer in New York,60.0,32.0,53.3
2,jsearch:Data Analyst in New York,76.0,42.0,55.3
3,theirstack,18.0,9.0,50.0


In [ ]:
# conn.close()
# print('Connection closed.')